# Imports

In [ ]:
from delta import configure_spark_with_delta_pip
from pathlib import Path
from custom_builder import builder
from log import *

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')


from pyspark.sql import functions as F

import json
from pathlib import Path

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/14 14:29:43 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/14 14:29:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2161922f-948a-4b4e-9136-24b1858a36b5;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Base taxi_trips queries

## Query 1

In [ ]:
with log_step("query_1_taxi_trips") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False)

## Query 2

In [ ]:
with log_step("query_2_taxi_trips") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

## Query 3

In [ ]:
with log_step("query_3_taxi_trips") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

# Create partitioned taxi_trips

In [ ]:
with log_step("create_taxi_trips_partitioned_by_date") as info:
    df = spark.read.format("delta").table("taxi_trips")

    df = df.withColumn("trip_date", F.to_date("pu_datetime"))

    df.write.format("delta") \
        .partitionBy("trip_date") \
        .mode("overwrite") \
        .saveAsTable("taxi_trips_partitioned_by_date")


In [ ]:
with log_step("create_taxi_trips_partitioned_by_county") as info:
    df = spark.read.format("delta").table("taxi_trips")

    new_df = spark.sql("""
            SELECT 
                pu_datetime, do_datetime, pu_location_id, do_location_id, fare_amount, county
            FROM default.taxi_trips t
            JOIN default.taxi_zone_lookup m
                ON t.pu_location_id = m.location_id
        """)

    new_df.show(5, truncate=False)

    new_df.write.format("delta") \
        .partitionBy("county") \
        .mode("overwrite") \
        .saveAsTable("taxi_trips_partitioned_by_county")

## Storage statistics

In [ ]:
# table details of partitioned tables
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").show(truncate=False)

# calculate the number of rows in each partitioned table # TODO: Figure out why number of rows differ
spark.sql("SELECT COUNT(*) AS row_count FROM taxi_trips_partitioned_by_date").show(truncate=False)
spark.sql("SELECT COUNT(*) AS row_count FROM taxi_trips_partitioned_by_county").show(truncate=False)

# numFiles of table on disk
spark.sql("DESCRIBE DETAIL taxi_trips").select("numFiles").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").select("numFiles").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").select("numFiles").show(truncate=False)

# sizeInBytes of table on disk
spark.sql("DESCRIBE DETAIL taxi_trips").select("sizeInBytes").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").select("sizeInBytes").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").select("sizeInBytes").show(truncate=False)


# Benchmark queries on partitioned tables

## Query 1

In [ ]:

with log_step("query_1_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips_partitioned_by_date t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False) 

In [ ]:
with log_step("query_1_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips_partitioned_by_county t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False) 

## Query 2

In [ ]:
with log_step("query_2_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips_partitioned_by_date
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

In [ ]:
with log_step("query_2_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips_partitioned_by_county
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

## Query 3

In [ ]:
with log_step("query_3_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips_partitioned_by_date t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

In [ ]:
with log_step("query_3_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips_partitioned_by_county t
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

# Drop partitioned tables

In [ ]:
spark.sql("DROP TABLE IF EXISTS taxi_trips_partitioned_by_date")
spark.sql("DROP TABLE IF EXISTS taxi_trips_partitioned_by_county")